# Next step: step 6b, choosing the start
**What it adds:** `sim` chooses when a run starts, from the passes it can see, instead of SPAM's search.

| | What it is |
|---|---|
| `sim.search_orbit(params, config)` | The path SPAM's search integrates: the secondary from now back to t = −30, one row per step (about 0.06 s) |
| `sim.find_passes(params, config)` | Every close pass on that path (a local minimum of the distance), most recent first |
| `SimulatorConfig(start=...)` | `"pass"` (new default), `"wallin"` (SPAM's search), or a negative number (start exactly then) |
| `SimulatorConfig(start_pass=...)` | With `"pass"`: which pass (1 = most recent, the default; 2, 3, …) |
| `SimulatorConfig(no_pass=...)` | With `"pass"`, when that pass doesn't exist back to −30: start here (−5.0, the default) or `"error"` |

With `"pass"`, the run starts `10 × rout1 / speed` before the chosen pass: SPAM's own lead (`sim.LEAD_FACTOR`),
so for pass 1 it matches SPAM's start whenever SPAM finds that pass. There's no −12 cap and no −5 fallback.

**Why:** 6a showed SPAM's search failing. This step shows why: these orbits are **periodic**, so the passes repeat
at the same distance, and "the closest pass in the window" is decided by rounding.

**How it connects:**
- **Below, in JSPAM:** `getTStart` keeps its backward path (`search_path`), and every run function takes
  `t_start`. When it's given, `setup_run` sets the start and SPAM's search doesn't run.
- **`sim`:** `_choose_start` turns the config into a `t_start` before every run: none for `"wallin"`, the number
  for a fixed start, or the chosen pass minus the lead.
- **Later:** a distance-based lead (some models are much faster than others), a configurable window.
- **Step 7 (many endings):** endings are measured from the start, so they build on this.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/setup_module.f90` (`getTStart`):** each step of the backward loop is saved (velocities flipped
back to forward time):
```fortran
allocate(search_path(int(-tmin / h) + 3, 7))
search_path_rows = 0
do while(tmin < t)
  search_path_rows = search_path_rows + 1   ! SIMR
  search_path(search_path_rows, 1:3) = rv(1:3)
  search_path(search_path_rows, 4:6) = -rv(4:6)
  search_path(search_path_rows, 7) = t
  ...
```
**JSPAM, `fortran/simr_interface.f90`:** a given start skips the search:
```fortran
subroutine set_start_time(t_start, use)      ! use = 1: start at t_start; 0: SPAM's search (the original)

! in setup_run
if (use_start_override) then
  tstart = start_override
  time = start_override
  tIsSet = .true.                            ! create_collision then doesn't call getTStart
end if

subroutine search_orbit(collision_param, potential, nmax, path, nrows)   ! returns search_path
```
**JSPAM, `src/jspam/__init__.py`:** `t_start=None` on `run_collision`, `initial_state`, `timing` and `run_orbit`
(set before every run, so it can't leak), and `search_orbit()`.

**`sim.py`:**
```python
LEAD_FACTOR = 10.0                      # SPAM's lead: 10 x rout1 / speed at the pass

start: str | float = "pass"             # new SimulatorConfig fields
start_pass: int = 1
no_pass: str | float = -5.0             # no such pass back to -30: start here, or "error"

def _passes(path):                      # local minima of the distance, most recent first
    distance = np.linalg.norm(path[:, :3], axis=1)
    rows = np.flatnonzero((distance[1:-1] < distance[:-2]) & (distance[1:-1] <= distance[2:])) + 1
    ...

def _choose_start(params, config):      # -> (t_start for jspam, the pass used)
    if config.start == "wallin":
        return None, None
    if not isinstance(config.start, str):
        return float(config.start), None
    chosen = find_passes(params, config)[config.start_pass - 1]     # missing: start at no_pass (-5)
    return chosen["t"] - LEAD_FACTOR * rout1 / chosen["speed"], chosen
```
`get_timing` also reports `"start"` (`"pass"`, `"wallin"` or `"fixed"`) and `"start_pass"`.

**Tests:** passes are found and ordered; pass 1's start matches SPAM's to a step; with `r_scale` 9, pass 1 is the
recent pass and pass 2 the one SPAM picked; a missing pass starts at −5 (or raises, with `no_pass="error"`), with the reason; a fixed start; every start lies exactly
on the search path; bad settings are rejected; the settings survive a JSON round trip. Older tests that check
SPAM's own search now ask for `start="wallin"`.

In [ ]:
%matplotlib inline
import time
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
config = sim.SimulatorConfig()
np.set_printoptions(suppress=True)


def plot_passes(ax, params, config, starts=None):
    """The distance between the centers back to -30, each pass numbered, and optional start lines."""
    search = sim.search_orbit(params, config)
    ax.plot(search[:, 6], np.linalg.norm(search[:, :3], axis=1), color="0.6", label="search path")
    for p in sim.find_passes(params, config):
        ax.plot(p["t"], p["distance"], "rs")
        ax.annotate(f"pass {p['pass']}", (p["t"], p["distance"]), textcoords="offset points", xytext=(0, -16),
                    ha="center", color="r")
    for i, (label, cfg) in enumerate((starts or {}).items()):
        t_start = sim.get_timing(params, cfg)["t_start"]
        ax.axvline(t_start, color=f"C{i}", ls="--", label=f"{label}: start {t_start:.2f}")
    ax.set_xlabel("time (SPAM units, now = 0)")
    ax.set_ylabel("distance between centers")
    ax.legend()

## 1. The search path
Row 0 is the model as given (now); each row after it is one step (0.001) further back:

In [ ]:
start_clock = time.perf_counter()
search = sim.search_orbit(params, config)
print(f"shape {search.shape}, in {time.perf_counter() - start_clock:.3f} s")
print("row 0:  ", search[0].round(3))
print("row -1: ", search[-1].round(3))

## 2. The passes
Two passes back to −30, **at the same distance and speed**: the orbit is bound and repeats. Friction, the only
thing that could change it, acts only inside r = 2, and these centers never get that close.

In [ ]:
for p in sim.find_passes(params, config):
    print(f"pass {p['pass']}: t = {p['t']:8.3f}   distance {p['distance']:.6f}   speed {p['speed']:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_passes(ax, params, config, {"pass 1": config, "pass 2": replace(config, start_pass=2),
                                 "SPAM's": replace(config, start="wallin")})
plt.show()

## 3. The start options
Pass 1's start matches SPAM's to within a step (both use the same lead on the same pass). Pass 2's starts before
−30: fine for a run (placement steps back on its own), though the search path doesn't reach it.

In [ ]:
options = {
    "pass 1 (default)": config,
    "pass 2": replace(config, start_pass=2),
    "SPAM's (wallin)": replace(config, start="wallin"),
    "fixed, -8": replace(config, start=-8.0),
}
print(f"{'':17s} {'t_start':>8s} {'n_steps':>7s} {'start':>7s} {'pass':>4s} {'t_closest':>9s}")
for label, cfg in options.items():
    tm = sim.get_timing(params, cfg)
    print(f"{label:17s} {tm['t_start']:8.3f} {tm['n_steps']:7d} {tm['start']:>7s} {str(tm['start_pass']):>4s} "
          f"{tm['t_closest']:9.3f}")

## 4. Where SPAM goes wrong: `r_scale` 9
Same model, slightly stronger force. Again two equal passes, but this time SPAM's "closest in the window" lands
on the older one (−23.4). Its start (−27.1) is before −12, so it falls back to −5.

In [ ]:
strong = sim.SimulatorConfig(r_scale=9.0)
starts = {
    "pass 1 (default)": strong,
    "pass 2": replace(strong, start_pass=2),
    "SPAM's (wallin)": replace(strong, start="wallin"),
}
for p in sim.find_passes(params, strong):
    print(f"pass {p['pass']}: t = {p['t']:8.3f}   distance {p['distance']:.6f}")
spam = sim.get_timing(params, starts["SPAM's (wallin)"])
print(f"\nSPAM's search: closest at {spam['t_closest']:.3f}, start found {spam['t_start_found']:.2f}, "
      f"used {spam['t_start']}, fallback {spam['start_fallback']}")

fig, ax = plt.subplots(figsize=(14, 5))
plot_passes(ax, params, strong, starts)
plt.show()

The particle runs. SPAM's −5 is still before pass 1 here (−2.87), so it shows that pass, but with a lead of
about 2.1 instead of 3.8, by luck. Pass 2's run starts before the older pass, so the disks go through both:

In [ ]:
N = 1500
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (label, cfg) in zip(axes, starts.items()):
    run = sim.simulate_particles(params, replace(cfg, n1=N, n2=N), seed=1)
    ax.scatter(run[:N, 0], run[:N, 1], s=1)
    ax.scatter(run[N:2 * N, 0], run[N:2 * N, 1], s=1)
    ax.set_title(f"{label}, start {sim.get_timing(params, cfg)['t_start']:.2f}")
    ax.set_aspect("equal")
plt.show()

## 5. Every start lies on the search path
The run places the secondary by stepping back from now along the same path, so its first row is a row of the
search path (the one `ceil(−t_start / 0.001)` steps back):

In [ ]:
for label, cfg in starts.items():
    first = sim.simulate_orbit(params, cfg)[0]
    row = int(np.ceil(-first[6] / 0.001 - 1e-9))
    path = sim.search_orbit(params, cfg)
    print(f"{label:17s} start {first[6]:7.3f}  row {row:5d}  largest difference {np.abs(first[:6] - path[row, :6]).max():.1e}")

## 6. When there's no pass to start from
Three ways a model can lack the pass asked for:
- **A first approach:** approaching now, with no earlier pass. There can't be tidal features yet.
- **A pass older than the window:** separating now, but the distance is still shrinking at t = −30.
- **Asking for more passes than there are.**

By default (`no_pass=-5.0`) the run starts at −5, close to what SPAM ends up doing, so the model still gets a
score an optimizer can rank. It isn't silent: `get_timing` reports `start="fallback"` and the reason.

In [ ]:
first = params.copy()
first[3:6] *= -3                      # reversed and faster: approaching, unbound
old = params.copy()
old[:3] *= 4                          # light, slow and distant: separating, with a pass before -30
old[3:6] *= 0.2
old[6:8] *= 0.1
cases = [("first approach", first, config), ("old pass", old, config),
         ("start_pass=5", params, replace(config, start_pass=5))]

for label, p, cfg in cases:
    tm = sim.get_timing(p, cfg)
    print(f"{label}: start {tm['t_start']} ({tm['start']})\n    reason: {tm['fallback_reason']}")

With `no_pass="error"` the same models raise a `ValueError` instead, before any particles are made:

In [ ]:
for label, p, cfg in cases:
    start_clock = time.perf_counter()
    try:
        sim.simulate_particles(p, replace(cfg, no_pass="error"), seed=1)
    except ValueError as error:
        print(f"{label} ({time.perf_counter() - start_clock:.3f} s) -> {error}\n")

And settings that can't work are caught when the config is made:

In [ ]:
for bad in [{"start": "later"}, {"start": 0.5}, {"start_pass": 0}, {"no_pass": "skip"}]:
    try:
        sim.SimulatorConfig(**bad)
    except ValueError as error:
        print(f"{bad} -> {error}")